# SI4006 · Sesión 10 · Lab: **RAG agéntico y evaluación con RAGAS**  ·  RESUELTO ✅

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

> 🔑 **Versión resuelta.** Los TODO vienen completos y cada bloque trae una nota de qué deberían ver.
> Las 👀 marcan lo que vale la pena mirar con calma.
>

Ya construimos un RAG que recupera bien. Hoy hacemos dos cosas encima: le damos **herramientas** para
actuar cuando la pregunta lo pide (tool use y ReAct), y aprendemos a **evaluarlo por dentro** con RAGAS,
que mira el contexto y no solo la respuesta.

- **Lab A · Function calling:** el modelo decide si llamar una herramienta (una calculadora o el corpus).
- **Lab B · Mini-agente ReAct:** el modelo piensa, actúa y observa, y repite hasta responder.
- **Lab C · RAGAS:** faithfulness, context precision/recall y answer relevancy, calculadas a mano.

> **Traigan su sistema de la S08.** Aquí levantamos un RAG compacto para que el notebook corra solo;
> donde diga, pueden pegar su propio retriever avanzado y su `eval_set`.

## 0 · Setup

In [ ]:
# Colab ya trae transformers y torch. Instalamos lo que falta.
%pip install -q sentence-transformers chromadb "opentelemetry-api==1.42.1" "opentelemetry-sdk==1.42.1"
print('Listo.')

In [ ]:
import torch, transformers, json, re
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('\n⚠️  En CPU esto va lento (el generador se llama muchas veces). Con T4 va cómodo.')

## 1 · Recap: un RAG compacto que corre solo

Reconstruimos rápido el pipeline (corpus → chunks → índice denso → generador → `sistema_rag`) y el
harness. Nada nuevo aquí; si vienen de la S08, es la misma tubería. Lo nuevo empieza en el Lab A.

In [ ]:
# Corpus semilla (dominio educación). Reemplácenlo por SU corpus real.
corpus = [
    {'id': 'doc1', 'fuente': 'Reglamento de evaluación (2026)',
     'texto': ('Las tareas para la casa valen como máximo el 15 por ciento de la nota del periodo, según '
               'el artículo 12. La entrega tardía se penaliza restando el 10 por ciento del puntaje de la '
               'tarea por cada día de retraso, hasta un máximo de tres días.')},
    {'id': 'doc2', 'fuente': 'Guía de matemáticas, fracciones (2026)',
     'texto': ('Para sumar fracciones con distinto denominador se busca el mínimo común múltiplo. Un error '
               'frecuente es sumar numeradores y denominadores por separado: 1/2 + 1/3 no es 2/5.')},
    {'id': 'doc3', 'fuente': 'Protocolo de acompañamiento (2026)',
     'texto': ('Cuando un estudiante presenta bajo rendimiento sostenido, el protocolo sugiere un plan de '
               'refuerzo de máximo seis semanas, con sesiones cortas de práctica espaciada.')},
]
print(len(corpus), 'documentos')

In [ ]:
# Chunking fixed-size con solapamiento.
def chunk_texto(t, tam=280, solapa=40):
    palabras = t.split(); out=[]; i=0
    while i < len(palabras):
        out.append(' '.join(palabras[i:i+tam])); i += tam - solapa
    return out
chunks=[]
for d in corpus:
    for j, ch in enumerate(chunk_texto(d['texto'])):
        chunks.append({'id': f"{d['id']}_c{j}", 'texto': ch, 'fuente': d['fuente']})
print(len(chunks), 'chunks')

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb, numpy as np
st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
cliente = chromadb.Client()
try: cliente.delete_collection('s10')
except Exception: pass
col = cliente.create_collection('s10', metadata={'hnsw:space': 'cosine'})
emb = st.encode([c['texto'] for c in chunks], show_progress_bar=False)
col.add(ids=[c['id'] for c in chunks], documents=[c['texto'] for c in chunks],
        metadatas=[{'fuente': c['fuente']} for c in chunks], embeddings=emb.tolist())
print('Índice denso listo:', col.count(), 'chunks')

In [ ]:
# Generador del curso.
from transformers import AutoModelForCausalLM, AutoTokenizer
GEN = 'Qwen/Qwen2.5-1.5B-Instruct'
gen_tok = AutoTokenizer.from_pretrained(GEN)
gen_model = AutoModelForCausalLM.from_pretrained(GEN, torch_dtype='auto').to(device)
def generar(system, user, max_new_tokens=220):
    msgs=[{'role':'system','content':system},{'role':'user','content':user}]
    prompt = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ent = gen_tok(prompt, return_tensors='pt').to(device)
    out = gen_model.generate(**ent, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=gen_tok.eos_token_id)
    return gen_tok.decode(out[0][ent['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print('Generador listo:', GEN)

In [ ]:
# RAG de una sola pasada (el punto de partida).
def buscar_semantica(consulta, k=3):
    e = st.encode([consulta]).tolist()
    r = col.query(query_embeddings=e, n_results=k)
    return list(zip(r['ids'][0], r['documents'][0], [m['fuente'] for m in r['metadatas'][0]]))
SYSTEM_RAG = ('Eres un asistente que responde SOLO con el contexto dado. Si la respuesta no está en el '
              'contexto, di: "No tengo esa información en mis fuentes."')
def sistema_rag(pregunta, k=3):
    ctx = buscar_semantica(pregunta, k)
    contexto = '\n\n'.join(f'[{fu}] {doc}' for _, doc, fu in ctx)
    return generar(SYSTEM_RAG, f'Contexto:\n{contexto}\n\nPregunta: {pregunta}', 160)
print(sistema_rag('¿cuánto valen las tareas?'))

In [ ]:
# Harness de evaluación (el mismo espíritu de M2, resumido).
eval_set = [
    {'input': '¿qué porcentaje máximo valen las tareas?', 'esperado': 'Como máximo el 15 por ciento de la nota del periodo.'},
    {'input': 'mi hijo suma arriba con arriba y abajo con abajo, ¿está bien?', 'esperado': 'No: sumar numeradores y denominadores por separado es un error.'},
    {'input': '¿cuántas horas de educación física exige el reglamento?', 'esperado': 'No tengo esa información en mis fuentes.'},
]
def sim(a, b):
    ea, eb = st.encode([a, b]); return float(np.dot(ea, eb) / (np.linalg.norm(ea)*np.linalg.norm(eb)))
def juez(preg, resp, esp):
    t = generar('Eres un evaluador estricto. Responde SOLO un número del 1 al 5.',
                f'Pregunta: {preg}\nReferencia: {esp}\nRespuesta: {resp}\nPuntaje (1-5):', 6)
    m = re.search(r'[1-5]', t); return int(m.group()) if m else 3
print('Harness y eval_set listos.')

## 2 · Lab A · Tool use: el modelo llama funciones

Le describimos herramientas con un esquema. El modelo, en vez de contestar siempre directo, puede pedir
que ejecutemos una: responde con un JSON `{"tool": ..., "args": {...}}`. Nosotros lo parseamos,
ejecutamos la función y le devolvemos el resultado como observación para que redacte la respuesta.

In [ ]:
# Dos herramientas: una calculadora y el corpus (nuestro RAG como tool).
def calculadora(expresion):
    try: return str(eval(expresion, {'__builtins__': {}}, {}))
    except Exception as e: return f'error: {e}'
def buscar_en_corpus(consulta):
    return '\n'.join(f'[{fu}] {doc}' for _, doc, fu in buscar_semantica(consulta, 3))
TOOLS = {'calculadora': calculadora, 'buscar_en_corpus': buscar_en_corpus}
ESQUEMAS = [
    {'name': 'calculadora', 'description': 'Evalúa una expresión aritmética y devuelve el número.', 'args': {'expresion': 'string'}},
    {'name': 'buscar_en_corpus', 'description': 'Busca en los documentos del colegio y devuelve pasajes.', 'args': {'consulta': 'string'}},
]
print('Herramientas:', list(TOOLS))

In [ ]:
SYSTEM_TOOLS = ('Tienes estas herramientas:\n' + json.dumps(ESQUEMAS, ensure_ascii=False, indent=2) +
  '\n\nSi necesitas una, responde SOLO con un JSON: {"tool": "nombre", "args": {...}}.'
  ' Si ya puedes responder, responde en texto normal, sin JSON.')

def _extraer_json(t):
    i, j = t.find('{'), t.rfind('}')
    if i != -1 and j != -1 and j > i:
        try: return json.loads(t[i:j+1])
        except Exception: return None
    return None

def ejecutar_tool(nombre, args):
    # Despacha: busca la función por nombre y la llama con los argumentos.
    fn = TOOLS.get(nombre)
    if fn is None: return f'error: no existe la herramienta {nombre}'
    return str(fn(**args))

In [ ]:
def responder_con_tools(pregunta, max_llamadas=3):
    historial = f'Pregunta: {pregunta}'
    for _ in range(max_llamadas):
        salida = generar(SYSTEM_TOOLS, historial, 160)
        pedido = _extraer_json(salida)
        if not pedido or 'tool' not in pedido:
            return salida            # el modelo respondió directo
        obs = ejecutar_tool(pedido['tool'], pedido.get('args', {}))
        print(f'  → llamó {pedido["tool"]}({pedido.get("args", {})}) = {obs}')
        historial += f'\nUsaste {pedido["tool"]} y obtuviste: {obs}\nAhora responde la pregunta.'
    return generar(SYSTEM_TOOLS, historial + '\nResponde ya, sin más herramientas.', 160)

print('P1:', responder_con_tools('¿cuánto es 15% de 40?'))
print('P2:', responder_con_tools('¿qué porcentaje máximo valen las tareas?'))

> 👀 **Lab A:** ante el cálculo exacto el modelo pidió `calculadora`; ante la pregunta de reglamento
> pidió `buscar_en_corpus`. Nosotros ejecutamos y le devolvimos el resultado. Si el JSON viene mal
> formado, `_extraer_json` devuelve `None` y el modelo responde directo: por eso conviene validar.

## 3 · Lab B · Un mini-agente ReAct

ReAct encadena pasos: el modelo escribe un **pensamiento**, elige una **acción** (una herramienta), lee
la **observación** y repite hasta poder responder. Así resuelve preguntas de varios pasos.

> **No todo necesita un agente.** Para una pregunta simple, el RAG de una pasada es mejor: menos latencia,
> menos costo y menos formas de fallar. El agente se justifica solo si mejora el resultado.

In [ ]:
SYSTEM_REACT = ('Resuelve la pregunta razonando por pasos. En cada paso escribe UNA línea así:\n'
  'Pensamiento: <tu razonamiento>\n'
  'Acción: buscar_en_corpus[consulta]  o  calculadora[expresion]  o  Responder[respuesta final]\n'
  'Usa Responder[...] cuando ya tengas la respuesta.')

def paso_react(historial):
    salida = generar(SYSTEM_REACT, historial, 120)
    linea = next((l for l in salida.splitlines() if 'Acción:' in l or 'Accion:' in l or 'Responder[' in l), salida)
    m = re.search(r'(buscar_en_corpus|calculadora|Responder)\[(.*?)\]', linea)
    return (salida, (m.group(1), m.group(2))) if m else (salida, None)

In [ ]:
def agente_react(pregunta, max_pasos=4):
    hist = f'Pregunta: {pregunta}'
    for _ in range(max_pasos):
        salida, accion = paso_react(hist)
        if accion is None:
            return salida
        nombre, arg = accion
        if nombre == 'Responder':
            return arg
        obs = ejecutar_tool('buscar_en_corpus' if nombre=='buscar_en_corpus' else 'calculadora',
                            {'consulta': arg} if nombre=='buscar_en_corpus' else {'expresion': arg})
        print(f'  → {nombre}[{arg}] = {obs[:80]}')
        hist += f'\nAcción: {nombre}[{arg}]\nObservación: {obs}'
    return 'No pude resolverlo en los pasos disponibles.'

print('SIMPLE  :', agente_react('¿qué porcentaje máximo valen las tareas?'))
print('COMPUESTA:', agente_react('si una tarea vale 40 puntos, ¿cuántos puntos son el máximo que puede valer en la nota?'))

> 👀 **Lab B:** en la compuesta el agente encadenó pasos (buscó el 15% y luego calculó); el RAG de una
> pasada no podía. En la simple dio más vueltas sin mejorar. Cada paso es una generación más: cuesta
> latencia. Ese es el criterio para decidir si vale un agente.

## 4 · Lab C · RAGAS: evaluar el RAG por dentro

RAGAS no es un flujo: toma un caso ya resuelto (pregunta, contexto, respuesta) y le calcula cuatro notas
de 0 a 1, cada una comparando dos piezas. Aquí las calculamos a mano con nuestro propio juez y embeddings,
para ver qué mide cada una. La librería `ragas` hace lo mismo por dentro (celda de referencia al final).

In [ ]:
# Armamos el caso de cada pregunta: (pregunta, contexto recuperado, respuesta, referencia).
casos = []
for e in eval_set:
    ctx = buscar_semantica(e['input'], 3)
    contexto = [doc for _, doc, _ in ctx]
    respuesta = sistema_rag(e['input'])
    casos.append({'pregunta': e['input'], 'contexto': contexto, 'respuesta': respuesta, 'referencia': e['esperado']})
print('Casos listos:', len(casos))

In [ ]:
def _si(t): return 1 if re.search(r'\bs[ií]\b|\byes\b|\b1\b', t.lower()) else 0

def faithfulness(caso):
    # Partimos la respuesta en afirmaciones y verificamos cada una contra el contexto.
    ctx = '\n'.join(caso['contexto'])
    afirmaciones = [s.strip() for s in re.split(r'[.\n]', caso['respuesta']) if len(s.strip()) > 12]
    if not afirmaciones: return 1.0
    ok = 0
    for a in afirmaciones:
        r = generar('Responde solo sí o no.', f'Contexto:\n{ctx}\n\n¿El contexto respalda esta afirmación? "{a}"', 4)
        ok += _si(r)
    return ok / len(afirmaciones)

def context_precision(caso):
    rel = 0
    for ch in caso['contexto']:
        r = generar('Responde solo sí o no.', f'Pregunta: {caso["pregunta"]}\n\n¿Este pasaje es relevante para responderla? "{ch}"', 4)
        rel += _si(r)
    return rel / max(1, len(caso['contexto']))

def context_recall(caso):
    ctx = '\n'.join(caso['contexto'])
    r = generar('Responde solo sí o no.', f'Contexto:\n{ctx}\n\n¿El contexto contiene lo necesario para llegar a esta referencia? "{caso["referencia"]}"', 4)
    return float(_si(r))

def answer_relevancy(caso):
    q2 = generar('Genera SOLO la pregunta que esta respuesta contestaría, sin nada más.', caso['respuesta'], 40)
    return sim(caso['pregunta'], q2)

In [ ]:
filas = []
for caso in casos:
    filas.append((caso['pregunta'][:34],
                  faithfulness(caso), context_precision(caso), context_recall(caso), answer_relevancy(caso)))
print(f'{"pregunta":<36}{"faith":>7}{"c.prec":>8}{"c.rec":>7}{"a.rel":>7}')
print('-'*65)
for f in filas:
    print(f'{f[0]:<36}{f[1]:>7.2f}{f[2]:>8.2f}{f[3]:>7.2f}{f[4]:>7.2f}')

### La librería `ragas` (referencia)

Para el reporte, pueden usar la librería en vez del cálculo a mano. Necesita un LLM y embeddings; por
defecto usa OpenAI, pero se le puede pasar un modelo local. La idea es la misma que vimos por dentro:

```python
# %pip install -q ragas datasets
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
# ds = Dataset.from_dict({'question': [...], 'answer': [...], 'contexts': [[...]], 'ground_truth': [...]})
# evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
```

## Lo que se llevan

Las métricas son un diagnóstico, no una nota que se persigue. Si context precision o recall salen bajas,
el problema es la recuperación (vuelvan a las técnicas de la S08). Si faithfulness sale baja, el modelo no
se apoya en el contexto (ajusten el prompt o quiten ruido). Si answer relevancy sale baja, la respuesta
divaga. Arreglen, vuelvan a medir y reporten estos números junto al scorecard del harness.

Y recuerden lo del agente: agregar pasos solo se justifica si el harness mejora de verdad.

*SI4006 · Universidad EAFIT · Sesión 10  ·  RESUELTO.*